In [ ]:
"""
Setup global paths for the project from environment variables.
"""
import os
# local
#CV_PATH_SPAIR = "./data/SPair71K"
# tfpool
CV_PATH_SPAIR = "/project/dl2026s/lmbcvtst/data/SPair71K"
os.environ["HF_HOME"] = "/project/dl2026s/lmbcvtst/data/hf_cache"


OUTPUT_DIR = "./results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image
import numpy as np
import torch
import glob
import json
import os

def read_img(path):
    img = np.array(Image.open(path).convert('RGB'))
    return torch.tensor(img.transpose(2, 0, 1).astype(np.float32))


class SPairDataset(Dataset):
    """Pairwise dataset from SPair71K returning source and target images with 2D keypoints."""

    def __init__(self, dataset_size='large', pck_alpha=0.1, datatype='test', img_size=256):
        pair_ann_path = CV_PATH_SPAIR + '/PairAnnotation'
        layout_path = CV_PATH_SPAIR + '/Layout'
        image_path = CV_PATH_SPAIR + '/JPEGImages'

        self.datatype = datatype
        self.pck_alpha = pck_alpha
        self.pair_ann_path = pair_ann_path
        self.image_path = image_path
        self.img_size = img_size
        self.categories = sorted(os.path.basename(x) for x in glob.glob('%s/*' % image_path))

        ann_files = open(os.path.join(layout_path, dataset_size, datatype + '.txt')).read().split('\n')
        self.ann_files = [a for a in ann_files if a]  # drop empty trailing line
        print(f'SPairDataset ({datatype}): {len(self.ann_files)} pairs')

    def _resize_img(self, img_tensor):
        """Resize shortest side to img_size then center-crop to img_size x img_size."""
        pil = transforms.ToPILImage()(img_tensor / 255.0)
        pil = transforms.Compose([
            transforms.Resize(self.img_size),
            transforms.CenterCrop(self.img_size),
        ])(pil)
        return transforms.ToTensor()(pil) * 255.0

    def _transform_kps(self, kps, imsize):
        """Map keypoints from original [W, H, C] imsize coords to resized/cropped coords."""
        W, H = imsize[0], imsize[1]
        scale = self.img_size / min(H, W)
        new_W, new_H = W * scale, H * scale
        x_off = (new_W - self.img_size) / 2
        y_off = (new_H - self.img_size) / 2
        transformed = []
        for kp in kps:
            x = kp[0] * scale - x_off
            y = kp[1] * scale - y_off
            transformed.append([x, y])
        return transformed

    def __len__(self):
        return len(self.ann_files)

    def _in_bounds(self, kps):
        """Boolean mask: True where both x and y are within [0, img_size)."""
        return ((kps[:, 0] >= 0) & (kps[:, 0] < self.img_size) &
                (kps[:, 1] >= 0) & (kps[:, 1] < self.img_size))

    def __getitem__(self, idx):
        ann_file = self.ann_files[idx] + '.json'
        with open(os.path.join(self.pair_ann_path, self.datatype, ann_file)) as f:
            annotation = json.load(f)

        category = annotation['category']
        src_img = self._resize_img(read_img(os.path.join(self.image_path, category, annotation['src_imname'])))
        trg_img = self._resize_img(read_img(os.path.join(self.image_path, category, annotation['trg_imname'])))

        src_kps = torch.tensor(self._transform_kps(annotation['src_kps'], annotation['src_imsize'])).float()
        trg_kps = torch.tensor(self._transform_kps(annotation['trg_kps'], annotation['trg_imsize'])).float()

        # Drop correspondences where either endpoint lies outside the cropped image
        valid = self._in_bounds(src_kps) & self._in_bounds(trg_kps)
        src_kps = src_kps[valid]
        trg_kps = trg_kps[valid]

        src_bbox = annotation['src_bndbox']
        pck_threshold = max(src_bbox[2] - src_bbox[0], src_bbox[3] - src_bbox[1]) * self.pck_alpha

        return {
            'pair_id': annotation['pair_id'],
            'src_imname': annotation['src_imname'],
            'trg_imname': annotation['trg_imname'],
            'category': category,
            'src_img': src_img,
            'trg_img': trg_img,
            'src_kps': src_kps,
            'trg_kps': trg_kps,
            'pck_threshold': pck_threshold,
        }


from torch.utils.data.dataloader import default_collate

def collate_var_len(batch):
    """Custom collate that keeps variable-length tensors as lists instead of stacking."""
    elem = batch[0]
    result = {}
    for key in elem:
        values = [d[key] for d in batch]
        if isinstance(values[0], torch.Tensor) and any(v.shape != values[0].shape for v in values):
            result[key] = values
        else:
            try:
                result[key] = default_collate(values)
            except RuntimeError:
                result[key] = values
    return result


print(f'SPAIR71K path: {CV_PATH_SPAIR}')
print(f'SPAIR71K exists: {os.path.exists(CV_PATH_SPAIR)}')

if os.path.exists(CV_PATH_SPAIR):
    spair_test = SPairDataset(datatype='test')
    sample = spair_test[0]
    print(f"\nFirst pair category: {sample['category']}")
    print(f"src: {sample['src_imname']}  shape: {sample['src_img'].shape}")
    print(f"trg: {sample['trg_imname']}  shape: {sample['trg_img'].shape}")
    print(f"src_kps: {sample['src_kps'].shape}, trg_kps: {sample['trg_kps'].shape}")


## Part 1 — Helper utilities and PCA

In this exercise you work with **patch-level features** from two vision foundation models:

- **DINO v2** (`facebook/dinov2-base`, ViT-B/14): produces 256 patches of size 16×16.  
  Each patch is represented by a 768-dimensional vector.
- **OpenCLIP** (`ViT-B-16`, laion2b): produces 196 patches of size 16×16.  
  Each patch is also 768-dimensional.

The features are already L2-normalised, so cosine similarity reduces to a dot product.

### What is PCK?

**Percentage of Correct Keypoints (PCK)** measures semantic correspondence quality:

1. For each annotated keypoint in the source image, find the source patch that contains it.
2. Compute the **nearest-neighbour patch** in the target image using cosine similarity.
3. A prediction is *correct* if the centre of the predicted patch lies within a threshold  
   distance of the ground-truth target keypoint (threshold = `α × max(bbox_width, bbox_height)`).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA
import torch.nn.functional as F

DINO_GRID = 16   # 256 patches → 16×16  (ViT-B/14, 224 px input)
CLIP_GRID = 14   # 196 patches → 14×14  (ViT-B/16, 224 px input)
PCA_DIMS  = [256, 128, 64, 32, 16]


def apply_pca(src_dict, trg_dict, n_components):
    """
    Fit PCA on all patch features (src + trg combined), then project and
    L2-normalise both feature dictionaries.

    Args:
        src_dict   : dict {img_name -> Tensor[num_patches, feat_dim]}
        trg_dict   : dict {img_name -> Tensor[num_patches, feat_dim]}
        n_components: int — number of PCA components to keep

    Returns:
        (src_reduced, trg_reduced) — same structure as input dicts but with
        each tensor having shape [num_patches, n_components] and L2-normalised rows.

    Steps to implement:
        1. Concatenate *all* patch features from both src_dict and trg_dict into
           a single NumPy array of shape [total_patches, feat_dim].
        2. Fit a PCA with `n_components` on this combined array (use random_state=0).
        3. Transform each feature tensor in src_dict and trg_dict with the fitted PCA.
        4. L2-normalise the reduced vectors (use F.normalize with dim=-1).
        5. Return the two transformed dicts (values as float32 Tensors).
    """
    # ── TODO: implement PCA fitting and projection ────────────────────────────
    combined_array = torch.cat([*src_dict.values(), *trg_dict.values()], dim=0).numpy()
    pca = PCA(n_components=n_components, random_state=0).fit(combined_array)
    src_reduced = {}
    for key, tensor in src_dict.items():
        transformed = pca.transform(tensor.numpy())  # NumPy array back
        normalized = F.normalize(torch.tensor(transformed).float(), dim=-1)
        src_reduced[key] = normalized
        
    trg_reduced = {}
    for key, tensor in trg_dict.items():
        transformed = pca.transform(tensor.numpy())  # NumPy array back
        normalized = F.normalize(torch.tensor(transformed).float(), dim=-1)
        trg_reduced[key] = normalized
        
    return (src_reduced, trg_reduced)  


def compute_pck(subset, src_dict, trg_dict, grid_size):
    """Compute PCK accuracy over all pairs in subset."""
    ps = 256 / grid_size
    correct, total = 0, 0
    for sample in subset:
        src_kps = sample['src_kps'].numpy()
        trg_kps = sample['trg_kps'].numpy()
        if len(src_kps) == 0:
            continue
        src_f  = src_dict[sample['src_imname']]
        trg_f  = trg_dict[sample['trg_imname']]
        nn_idx = (src_f @ trg_f.T).argmax(dim=1)
        pck_thr = sample['pck_threshold']
        for (kx, ky), (gtx, gty) in zip(src_kps, trg_kps):
            pj = int(np.clip(kx / ps, 0, grid_size - 1))
            pi = int(np.clip(ky / ps, 0, grid_size - 1))
            ni, nj = divmod(nn_idx[pi * grid_size + pj].item(), grid_size)
            pred_x, pred_y = (nj + 0.5) * ps, (ni + 0.5) * ps
            correct += int(np.sqrt((pred_x - gtx) ** 2 + (pred_y - gty) ** 2) < pck_thr)
            total += 1
    return correct / total if total > 0 else 0.0, correct, total


def make_pca_image(feats, grid_size, pca3, vmin, vmax):
    """
    Render patch features as a 256×256 RGB image using the first 3 PCA components.
    vmin / vmax: per-channel bounds (shape [3,]) for globally consistent colouring.
    """
    comps = pca3.transform(feats.numpy())               # [P, 3]
    comps = ((comps - vmin) / (vmax - vmin + 1e-8)).clip(0, 1)
    img_small = (comps.reshape(grid_size, grid_size, 3) * 255).astype(np.uint8)
    return np.array(Image.fromarray(img_small).resize((256, 256), Image.NEAREST)) / 255.0


def draw_nn_correspondences(ax, sample, src_feats, trg_feats, grid_size, title='',
                             src_img=None, trg_img=None):
    """
    Draw NN patch correspondences on ax.
    src_img / trg_img: optional (H, W, 3) float [0,1] overrides; default = sample images.
    Returns (n_correct, n_total).
    """
    if src_img is None:
        src_img = (sample['src_img'].permute(1, 2, 0).numpy() / 255.0).clip(0, 1)
    if trg_img is None:
        trg_img = (sample['trg_img'].permute(1, 2, 0).numpy() / 255.0).clip(0, 1)

    H, W    = src_img.shape[:2]
    ps      = H / grid_size
    canvas  = np.concatenate([src_img, trg_img], axis=1)
    ax.imshow(canvas)
    ax.axis('off')

    src_kps = sample['src_kps'].numpy()
    trg_kps = sample['trg_kps'].numpy()
    pck_thr = sample['pck_threshold']

    if len(src_kps) == 0:
        ax.set_title(f"{title}  [no kps]", fontsize=7, pad=2)
        return 0, 0

    nn_idx = (src_feats @ trg_feats.T).argmax(dim=1)
    colors = plt.cm.hsv(np.linspace(0, 1, len(src_kps), endpoint=False))
    n_correct = 0
    for ci, ((kx, ky), (gtx, gty)) in enumerate(zip(src_kps, trg_kps)):
        pj = int(np.clip(kx / ps, 0, grid_size - 1))
        pi = int(np.clip(ky / ps, 0, grid_size - 1))
        ni, nj  = divmod(nn_idx[pi * grid_size + pj].item(), grid_size)
        pred_x  = (nj + 0.5) * ps
        pred_y  = (ni + 0.5) * ps
        correct = np.sqrt((pred_x - gtx) ** 2 + (pred_y - gty) ** 2) < pck_thr
        n_correct += int(correct)
        c  = colors[ci]
        tx = pred_x + W
        ax.plot([kx, tx], [ky, pred_y], '-', color=c, lw=0.9, alpha=0.7)
        ax.plot(kx, ky, 'o', color=c, markersize=5)
        if correct:
            ax.plot(tx, pred_y, 'o', color=c, markersize=6,
                    markeredgewidth=1.2, markeredgecolor='white')
        else:
            ax.plot(tx, pred_y, 'x', color=c, markersize=8, markeredgewidth=2.0)

    ax.set_title(f"{title}  [{n_correct}/{len(src_kps)}]", fontsize=7, pad=2)
    return n_correct, len(src_kps)


def make_pca_image_component(feats, grid_size, pca6, vmin_per_comp, vmax_per_comp, component_idx):
    """
    Render a single PCA component (0-indexed) as a viridis-coloured 256x256 RGB image.
    vmin_per_comp / vmax_per_comp: per-component global bounds (shape [6,]).
    """
    import matplotlib.cm as cm
    comps = pca6.transform(feats.numpy())[:, component_idx]  # [P]
    vmin  = vmin_per_comp[component_idx]
    vmax  = vmax_per_comp[component_idx]
    norm  = ((comps - vmin) / (vmax - vmin + 1e-8)).clip(0, 1).reshape(grid_size, grid_size)
    rgb   = cm.viridis(norm)[:, :, :3]
    return np.array(Image.fromarray((rgb * 255).astype('uint8')).resize((256, 256), Image.NEAREST)) / 255.0


def make_pca_image_triplet(feats, grid_size, pca6, vmin6, vmax6, start):
    """
    Render patch features as a 256x256 RGB image using 3 consecutive PCA components.
    start: 0 for components 1-3, 3 for components 4-6.
    vmin6 / vmax6: per-component global bounds (shape [6,]).
    """
    comps = pca6.transform(feats.numpy())[:, start:start + 3]  # [P, 3]
    vmin  = vmin6[start:start + 3]
    vmax  = vmax6[start:start + 3]
    comps = ((comps - vmin) / (vmax - vmin + 1e-8)).clip(0, 1)
    img_small = (comps.reshape(grid_size, grid_size, 3) * 255).astype(np.uint8)
    return np.array(Image.fromarray(img_small).resize((256, 256), Image.NEAREST)) / 255.0


In [ ]:
import torch.nn.functional as F
import open_clip
from transformers import AutoImageProcessor, AutoModel
from tqdm import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

# Persistent cache within the session: (model_type, img_name) -> [num_patches, feat_dim]
feature_cache = {}

@torch.no_grad()
def extract_features(samples, model_type='dino', batch_size=32):
    """
    Extract patch-level features for a list of dataset samples, with caching by image name.

    Each sample must have:
        'src_img'   : float tensor [C, H, W] with pixel values in [0, 255]
        'src_imname': str used as cache key (features are not re-extracted if already cached)

    Returns:
        dict: {img_name -> tensor [num_patches, feat_dim]}
              DINO ViT-B/16: num_patches=256
              CLIP ViT-B/16: num_patches=196
    """
    def cache_key(name):
        return (model_type, name)

    uncached = [s for s in samples if cache_key(s['src_imname']) not in feature_cache]

    if uncached:
        if model_type == 'dino':
            processor = AutoImageProcessor.from_pretrained('facebook/dinov2-base')
            backbone = AutoModel.from_pretrained('facebook/dinov2-base').to(DEVICE).eval()
        elif model_type == 'clip':
            clip_model, _, preprocess = open_clip.create_model_and_transforms(
                'ViT-B-16', pretrained='laion2b_s34b_b88k')
            clip_model = clip_model.to(DEVICE).eval()
        else:
            raise ValueError(f"Unknown model_type: {model_type!r}")

        def to_pil(img):
            img_f = img.float()
            if img_f.max() > 1.0:
                img_f = img_f / 255.0
            return transforms.ToPILImage()(img_f.clamp(0, 1))

        for i in tqdm(range(0, len(uncached), batch_size), desc=f'{model_type} patch features'):
            batch   = uncached[i:i + batch_size]
            names   = [s['src_imname'] for s in batch]
            pil_imgs = [to_pil(s['src_img']) for s in batch]

            if model_type == 'dino':
                inputs = processor(pil_imgs, return_tensors='pt')
                output = backbone(inputs['pixel_values'].to(DEVICE))
                # last_hidden_state: [B, 1+num_patches, D]; index 0 is the CLS token
                patch_feats = output.last_hidden_state[:, 1:, :]  # [B, num_patches, D]

            else:  # clip
                pixels = torch.stack([preprocess(img) for img in pil_imgs]).to(DEVICE)
                # Hook the last transformer block to capture tokens before the CLS-only projection
                captured = []
                hook = clip_model.visual.transformer.resblocks[-1].register_forward_hook(
                    lambda m, inp, out: captured.append(out))
                clip_model.encode_image(pixels)
                hook.remove()
                # captured[0]: [B, 1+num_patches, C] — index 0 is the CLS token
                patch_feats = clip_model.visual.ln_post(captured[0][:, 1:])  # [B, num_patches, C]

            patch_feats = F.normalize(patch_feats, dim=-1).cpu()
            for name, feats in zip(names, patch_feats):
                feature_cache[cache_key(name)] = feats

    return {s['src_imname']: feature_cache[cache_key(s['src_imname'])] for s in samples}


## Task 1 — Implement `apply_pca` above, then run feature extraction and PCA fitting

Go back to the cell above and implement the body of `apply_pca`.

**Hints:**
- Use `torch.cat(list(d.values()), dim=0).numpy()` to stack all patch tensors.
- `sklearn.decomposition.PCA` accepts a NumPy array; use `.fit()` then `.transform()`.
- After projecting, wrap the result in `torch.tensor(...).float()` and apply `F.normalize(…, dim=-1)`.
- The fitting must be done on **src and trg combined** so both dictionaries land in the same PCA space.

Once implemented, execute the cell below to sample pairs, extract features, and fit PCA for all dimensionalities.

In [ ]:
import random
from collections import defaultdict

random.seed(42)
N_PER_CAT = 10

# Group test-set pairs by category
cat_to_indices = defaultdict(list)
for i, ann_file in enumerate(spair_test.ann_files):
    cat_to_indices[ann_file.split(':')[1]].append(i)

categories = sorted(cat_to_indices.keys())
cat_samples = {
    cat: [spair_test[idx] for idx in random.sample(indices, min(N_PER_CAT, len(indices)))]
    for cat, indices in cat_to_indices.items()
}

print(f"{len(categories)} categories, up to {N_PER_CAT} pairs each:")
for cat in categories:
    print(f"  {cat:<16}: {len(cat_samples[cat])} pairs")

# Collect all unique images across every category
src_set, trg_set = {}, {}
for samples in cat_samples.values():
    for s in samples:
        src_set[s['src_imname']] = s['src_img']
        trg_set[s['trg_imname']] = s['trg_img']

src_all_list = [{'src_img': img, 'src_imname': name} for name, img in src_set.items()]
trg_all_list = [{'src_img': img, 'src_imname': name} for name, img in trg_set.items()]
print(f"\nUnique images: {len(src_set)} src, {len(trg_set)} trg")

print("\nExtracting DINO features...")
dino_src_all = extract_features(src_all_list, model_type='dino', batch_size=32)
dino_trg_all = extract_features(trg_all_list, model_type='dino', batch_size=32)

print("\nExtracting CLIP features...")
clip_src_all = extract_features(src_all_list, model_type='clip', batch_size=32)
clip_trg_all = extract_features(trg_all_list, model_type='clip', batch_size=32)

# Fit PCA for all dimensionalities used in the accuracy table
print("\nFitting PCA for accuracy table...")
pca_dino_all, pca_clip_all = {}, {}
for n in PCA_DIMS:
    pca_dino_all[n] = apply_pca(dino_src_all, dino_trg_all, n)
    pca_clip_all[n] = apply_pca(clip_src_all, clip_trg_all, n)
    print(f"  PCA-{n} done")

# ── Task 2: Fit PCA-6 for visualisation ────────────────────────────────────────
# Use only the two animal categories below (they share enough visual structure
# to give a meaningful low-dimensional colour space).
PCA_VIZ_CATS = ['cat', 'dog']

# TODO: collect all patch features from the source and target images of
#       PCA_VIZ_CATS (using dino_src_all / dino_trg_all and
#       clip_src_all / clip_trg_all), stack them into two NumPy arrays
#       (one per model), and fit a PCA(n_components=6, random_state=0)
#       on each.  Also compute the per-component min/max across all patches
#       (needed for consistent colour scaling in the visualisations below).
#
# You need to define:
#   pca6_dino, pca6_clip       — fitted sklearn PCA objects (6 components)
#   dino6_vmin, dino6_vmax     — shape [6,] arrays: per-component global min/max
#   clip6_vmin, clip6_vmax     — same for CLIP
#
# Hint: PCA.transform(array).min(0) gives the per-component minimum.
all_dino_feats = []
all_clip_feats = []

for cat in PCA_VIZ_CATS:
    samples = cat_samples[cat]
    
    for sample in samples:
        src_name = sample['src_imname']
        trg_name = sample['trg_imname']
        
        all_dino_feats.append(dino_src_all[src_name])
        all_dino_feats.append(dino_trg_all[trg_name])
        
        all_clip_feats.append(clip_src_all[src_name])
        all_clip_feats.append(clip_trg_all[trg_name])

# Combine into NumPy arrays
dino_combined = torch.cat(all_dino_feats, dim=0).numpy()
clip_combined = torch.cat(all_clip_feats, dim=0).numpy()

# Fit PCA-6
pca6_dino = PCA(n_components=6, random_state=0).fit(dino_combined)
pca6_clip = PCA(n_components=6, random_state=0).fit(clip_combined)

# Compute per-component min/max
dino_transformed = pca6_dino.transform(dino_combined)
clip_transformed = pca6_clip.transform(clip_combined)

dino6_vmin = dino_transformed.min(axis=0)
dino6_vmax = dino_transformed.max(axis=0)

clip6_vmin = clip_transformed.min(axis=0)
clip6_vmax = clip_transformed.max(axis=0)    

print("Done.")


In [ ]:
import pandas as pd

# ── Table 1: Full-feature PCK per category ────────────────────────────────────
rows_cat = {'DINO': {}, 'CLIP': {}}
for cat in categories:
    samples = cat_samples[cat]
    acc_d, *_ = compute_pck(samples, dino_src_all, dino_trg_all, DINO_GRID)
    acc_c, *_ = compute_pck(samples, clip_src_all, clip_trg_all, CLIP_GRID)
    rows_cat['DINO'][cat] = acc_d * 100
    rows_cat['CLIP'][cat] = acc_c * 100

df1 = pd.DataFrame(rows_cat).T
df1['Mean'] = df1.mean(axis=1)

print("Table 1 — PCK Accuracy (%) per Category  [Full Features]")
print("=" * 80)
with pd.option_context('display.float_format', '{:.1f}'.format,
                       'display.max_columns', None, 'display.width', 300):
    print(df1.to_string())


out_path = f"{OUTPUT_DIR}/table1_pck_per_category.json"
df1.to_json(out_path, indent=2, orient="index")
print(f"Saved to {out_path}")


In [ ]:
# ── Table 2: PCK across feature dimensionalities (mean over all categories) ───
all_flat = [s for slist in cat_samples.values() for s in slist]

rows_pca = {'DINO': {}, 'CLIP': {}}

acc_d, *_ = compute_pck(all_flat, dino_src_all, dino_trg_all, DINO_GRID)
acc_c, *_ = compute_pck(all_flat, clip_src_all, clip_trg_all, CLIP_GRID)
rows_pca['DINO']['Full'] = acc_d * 100
rows_pca['CLIP']['Full'] = acc_c * 100

for n in sorted(PCA_DIMS, reverse=True):   # 256 → 128 → 64 → 32 → 16
    ps_d, pt_d = pca_dino_all[n]
    ps_c, pt_c = pca_clip_all[n]
    acc_d, *_ = compute_pck(all_flat, ps_d, pt_d, DINO_GRID)
    acc_c, *_ = compute_pck(all_flat, ps_c, pt_c, CLIP_GRID)
    rows_pca['DINO'][f'PCA-{n}'] = acc_d * 100
    rows_pca['CLIP'][f'PCA-{n}'] = acc_c * 100

col_order = ['Full'] + [f'PCA-{n}' for n in sorted(PCA_DIMS, reverse=True)]
df2 = pd.DataFrame(rows_pca).T[col_order]

print("Table 2 — PCK Accuracy (%) by Feature Dimensionality  [All Categories]")
print("=" * 60)
with pd.option_context('display.float_format', '{:.1f}'.format):
    print(df2.to_string())


out_path = f"{OUTPUT_DIR}/table2_pck_by_dim.json"
df2.to_json(out_path, indent=2, orient="index")
print(f"Saved to {out_path}")


In [ ]:
# ── Visualisation 1: 5 selected pairs, full features, RGB images ──────────────
VIZ_CATEGORIES = ['cat', 'dog']
random.seed(0)
viz_samples = {cat: random.choice(cat_samples[cat]) for cat in VIZ_CATEGORIES}

fig, axes = plt.subplots(len(VIZ_CATEGORIES), 2, figsize=(18, 4 * len(VIZ_CATEGORIES)))
for row, cat in enumerate(VIZ_CATEGORIES):
    sample = viz_samples[cat]
    sn, tn = sample['src_imname'], sample['trg_imname']
    draw_nn_correspondences(axes[row, 0], sample,
                            dino_src_all[sn], dino_trg_all[tn], DINO_GRID,
                            title=f"DINO | {cat} | {sn} → {tn}")
    draw_nn_correspondences(axes[row, 1], sample,
                            clip_src_all[sn], clip_trg_all[tn], CLIP_GRID,
                            title=f"CLIP | {cat} | {sn} → {tn}")

plt.suptitle("NN Correspondences — Full Features (RGB images)", fontsize=13, y=1.002)
plt.tight_layout()
out_path = f"{OUTPUT_DIR}/viz_full_rgb.png"
plt.savefig(out_path, dpi=100, bbox_inches='tight')
plt.show()
print(f"Saved to {out_path}")


In [ ]:
def _render_pca_triplet(components):
    """Visualise the first or last 3 PCA-6 components as RGB for all categories."""
    start = 0 if components == 'Components 1–3' else 3
    label = components
    fig, axes = plt.subplots(len(VIZ_CATEGORIES), 2, figsize=(18, 4 * len(VIZ_CATEGORIES)))
    for row, cat in enumerate(VIZ_CATEGORIES):
        sample = viz_samples[cat]
        sn, tn = sample['src_imname'], sample['trg_imname']
        src_rgb = (sample['src_img'].permute(1, 2, 0).numpy() / 255.0).clip(0, 1)
        trg_rgb = (sample['trg_img'].permute(1, 2, 0).numpy() / 255.0).clip(0, 1)

        dino_src_img = 0.3 * src_rgb + 0.7 * make_pca_image_triplet(
            dino_src_all[sn], DINO_GRID, pca6_dino, dino6_vmin, dino6_vmax, start)
        dino_trg_img = 0.3 * trg_rgb + 0.7 * make_pca_image_triplet(
            dino_trg_all[tn], DINO_GRID, pca6_dino, dino6_vmin, dino6_vmax, start)
        clip_src_img = 0.3 * src_rgb + 0.7 * make_pca_image_triplet(
            clip_src_all[sn], CLIP_GRID, pca6_clip, clip6_vmin, clip6_vmax, start)
        clip_trg_img = 0.3 * trg_rgb + 0.7 * make_pca_image_triplet(
            clip_trg_all[tn], CLIP_GRID, pca6_clip, clip6_vmin, clip6_vmax, start)

        draw_nn_correspondences(axes[row, 0], sample,
                                dino_src_all[sn], dino_trg_all[tn], DINO_GRID,
                                title=f'DINO | {cat} | PCA-6 {label}',
                                src_img=dino_src_img, trg_img=dino_trg_img)
        draw_nn_correspondences(axes[row, 1], sample,
                                clip_src_all[sn], clip_trg_all[tn], CLIP_GRID,
                                title=f'CLIP | {cat} | PCA-6 {label}',
                                src_img=clip_src_img, trg_img=clip_trg_img)

    plt.suptitle(
        f'NN Correspondences — PCA-6 feature image ({label} → RGB)',
        fontsize=13, y=1.002)
    plt.tight_layout()
    out_path = f'{OUTPUT_DIR}/viz_pca6_{"first3" if start == 0 else "last3"}.png'
    plt.savefig(out_path, dpi=100, bbox_inches='tight')
    plt.show()
    print(f'Saved to {out_path}')

_render_pca_triplet('Components 1–3')
_render_pca_triplet('Components 4–6')